In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 3, Finished, Available, Finished, False)

In [4]:
DeltaTable.createIfNotExists(spark)\
          .tableName("dim.Gold_OrderPriority")\
          .addColumn("OrderPriority_ID",LongType())\
          .addColumn("Order_Priority",StringType())\
          .addColumn("Created_TS",TimestampType())\
          .addColumn("Modified_TS",TimestampType())\
          .execute()

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 6, Finished, Available, Finished, False)

In [5]:
df=spark.read.table("LH_Silver.dim.gold_orderpriority")

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 7, Finished, Available, Finished, False)

In [6]:
Max_Date=df.selectExpr("coalesce(max(Modified_TS),'1900-01-01')").first()[0]

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 8, Finished, Available, Finished, False)

In [7]:
Max_Date

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 9, Finished, Available, Finished, False)

'1900-01-01'

In [8]:
df_bronze=spark.read.table("Lakehouse_Bronze.dbo.bronze_sales")

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 10, Finished, Available, Finished, False)

In [9]:
df_bronze_mod=df_bronze.select("Order_Priority").where(col("Modified_TS")>Max_Date).drop_duplicates()

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 11, Finished, Available, Finished, False)

In [10]:
Max_ID=df.selectExpr("coalesce(max(OrderPriority_ID),0)").first()[0]

df_final=df_bronze_mod.withColumn("OrderPriority_ID",Max_ID+monotonically_increasing_id()+1)

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 12, Finished, Available, Finished, False)

In [12]:
df_gold_delta=DeltaTable.forPath(spark,"Tables/dim/gold_orderpriority")
df_bonze_table=df_final

StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 14, Finished, Available, Finished, False)

In [13]:
df_gold_delta.alias("gold")\
             .merge(\
               df_bonze_table.alias("bronze"),\
                "gold.Order_Priority==Bronze.Order_Priority"\
             )\
             .whenMatchedUpdate(\
                  set={
                    "gold.Modified_TS":current_timestamp()
                  }
             )\
             .whenNotMatchedInsert(\
                    values={
                         "gold.OrderPriority_ID":"Bronze.OrderPriority_ID",
                     "gold.Order_Priority":"Bronze.Order_Priority",
                     "gold.Created_TS":current_timestamp(),
                     "gold.Modified_TS":current_timestamp()
                    }
             )\
             .execute()


StatementMeta(, a1f85433-b2b4-440f-b1ba-2b33877e2be0, 15, Finished, Available, Finished, False)